In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import detrend, savgol_filter, butter, filtfilt
from scipy.fft import rfft, rfftfreq
from scipy.optimize import minimize

# ============================================================
# LOAD DATA
# ============================================================

def load_data():

    lift = pd.read_csv(
        "drag_lift_body_001.dat",
        sep=r"\s+",
        skiprows=1,
        names=["time","cxp","cxs","cx","cyp","cys","cy"]
    )

    struct = pd.read_csv(
        "viv_body1_probe.dat",
        sep=r"\s+",
        skiprows=2,
        names=["ntime","time","DX","DY","VY","AY"]
    )

    t = struct.time.values
    y = struct.DY.values

    # remove transient (wait for flow to develop)
    mask = t > 50.0
    t = t[mask]
    y = y[mask]

    # interpolate total lift coefficient onto the structural time grid
    C_y = np.interp(t, lift.time.values, lift.cy.values)

    t = t - t[0]

    return t, y, C_y


# ============================================================
# FREQUENCY EXTRACTION
# ============================================================

def compute_frequency(t, y):

    y = detrend(y)

    dt = np.mean(np.diff(t))
    N = len(t)

    Y = rfft(y)
    freqs = rfftfreq(N, dt)

    Y[0] = 0

    idx = np.argmax(np.abs(Y))
    f = freqs[idx]

    omega = 2*np.pi*f
    T = 1/f

    return f, omega, T


# ============================================================
# LOWPASS FILTER (PRESERVES NONLINEARITIES)
# ============================================================

def lowpass(x, fs, f0):

    # Lowpass filter at 5 times the fundamental frequency to keep nonlinear harmonics
    # while removing high frequency numerical noise
    cutoff = 5.0 * f0
    b, a = butter(4, cutoff / (fs/2), btype='low')
    return filtfilt(b, a, x)


# ============================================================
# IDENTIFICATION (ROBUST NONLINEAR FOR D-SECTION)
# ============================================================

def identify_combined_params(t, y, C_y, omega):

    dt = np.mean(np.diff(t))

    # Calculate structural velocity and acceleration directly
    y_t  = savgol_filter(y, window_length=101, polyorder=3, deriv=1, delta=dt)
    y_tt = savgol_filter(y, window_length=101, polyorder=3, deriv=2, delta=dt)

    # Detrend lift to ensure zero mean
    C_y = detrend(C_y)

    CL0 = 0.328

    def objective(params):
        eps, A, a1, a3 = params

        # Decompose the total lift coefficient:
        # C_y(total) = C_VIV + C_Gal
        # where C_Gal (galloping) is modeled via a quasi-steady polynomial of structural velocity.
        C_Gal = a1 * y_t + a3 * (y_t**3)
        C_VIV = C_y - C_Gal

        # Relate VIV lift to the wake oscillator variable q
        q = 2 * C_VIV / CL0

        # Derivatives of q
        q_t  = savgol_filter(q, window_length=101, polyorder=3, deriv=1, delta=dt)
        q_tt = savgol_filter(q, window_length=101, polyorder=3, deriv=2, delta=dt)

        # Residual of the Facchinetti wake oscillator equation
        # q'' + eps * omega * (q^2 - 1) * q' + omega^2 * q = A * y''
        residual = q_tt + eps * omega * (q**2 - 1) * q_t + (omega**2) * q - A * y_tt

        # Objective is to minimize the mean squared error
        return np.mean(residual**2)

    # Initial guesses:
    # [eps, A] roughly based on literature for circular cylinders (start point)
    # [a1, a3] = [0, 0] assuming purely VIV at first. The optimizer will adjust.
    initial_guess = [0.1, 12.0, 1.0, -10.0]

    # Run bounded optimization
    res = minimize(objective, initial_guess, method='L-BFGS-B', options={'disp': False})

    eps_opt, A_opt, a1_opt, a3_opt = res.x
    return eps_opt, A_opt, a1_opt, a3_opt


# ============================================================
# MAIN
# ============================================================

def main():

    print("\n===== D-SECTION VIV & GALLOPING IDENTIFICATION =====")

    t, y, C_y = load_data()

    # --------------------------------------------------------
    # MASS & DAMPING
    # --------------------------------------------------------
    # Parameters given by standard low-mass-ratio FIV literature (e.g. Sharma et al., 2022)
    m_total = 5.0
    zeta = 0.01

    # --------------------------------------------------------
    # FREQUENCY
    # --------------------------------------------------------
    f, omega, T = compute_frequency(t, y)

    # --------------------------------------------------------
    # FILTER SIGNALS
    # --------------------------------------------------------
    fs = 1 / np.mean(np.diff(t))

    y   = lowpass(y, fs, f)
    C_y = lowpass(C_y, fs, f)

    # --------------------------------------------------------
    # STRUCTURAL PARAMETERS
    # --------------------------------------------------------
    k = m_total * omega**2
    c = 2 * zeta * m_total * omega

    # --------------------------------------------------------
    # IDENTIFICATION
    # --------------------------------------------------------
    # We jointly identify wake parameters (eps, A) and galloping coefficients (a1, a3)
    eps, A, a1, a3 = identify_combined_params(t, y, C_y, omega)

    # ========================================================
    # OUTPUT AND VERIFICATION
    # ========================================================

    print("\n================ RESULTS & VERIFICATION ================")
    print("Kinematics:")
    print(f" Frequency f       : {f:.5f} Hz")
    print(f" Angular freq ω    : {omega:.5f} rad/s")
    print(f" Period T          : {T:.5f} s")
    print("--------------------------------------------------------")
    print("Structural Properties:")
    print(f" Mass (m)          : {m_total:.5f}   [Expected: ~5.0 (Standard low mass ratio)]")
    print(f" Damping Ratio (ζ) : {zeta:.5f}   [Expected: ~0.01 (Standard low damping)]")
    print(f" Stiffness (k)     : {k:.5f}   [Derived from m * ω^2]")
    print(f" Damping (c)       : {c:.5f}   [Derived from 2 * ζ * m * ω]")
    print("--------------------------------------------------------")
    print("Wake Oscillator Parameters (Facchinetti VIV Model):")
    print(f" Epsilon (ε)       : {eps:.5f}   [Expected: ~0.05 to 0.5. Controls nonlinear saturation]")
    print(f" Coupling (A)      : {A:.5f}  [Expected: ~5 to 20. Scales structural acceleration effect]")
    print("--------------------------------------------------------")
    print("Galloping Lift Coefficients (Quasi-Steady Polynomial):")
    print(f" a1 (Linear)       : {a1:.5f}   [Expected: > 0. Positive value triggers soft galloping]")
    print(f" a3 (Cubic)        : {a3:.5f} [Expected: < 0. Negative value ensures stable limit cycle]")
    print("========================================================")

    print("\n================ GOVERNING EQUATIONS =================")

    print("\n1. Structure Equation:")
    print(f"   {m_total:.3f} y'' + {c:.3f} y' + {k:.3f} y = 1/2 ρ U^2 D C_y(t)")

    print("\n2. Lift Decomposition (Combined VIV & Galloping):")
    print(f"   C_y(t) = C_VIV(t) + {a1:.3f} y' + {a3:.3f} (y')^3")

    print("\n3. Wake Oscillator (Governing the C_VIV component):")
    print(f"   q'' + {eps:.4f} * {omega:.3f} * (q^2 - 1)q' + {omega**2:.3f} q = {A:.3f} y''")
    print(f"   where C_VIV = (0.328 / 2) * q")

    print("\n=====================================================\n")

if __name__ == "__main__":
    main()


===== D-SECTION VIV & GALLOPING IDENTIFICATION =====


/tmp/ipykernel_1247/3510181500.py:127: DeprecationWarning: scipy.optimize: The `disp` and `iprint` options of the L-BFGS-B solver are deprecated and will be removed in SciPy 1.18.0.
  res = minimize(objective, initial_guess, method='L-BFGS-B', options={'disp': False})



================ RESULTS & VERIFICATION ================
Kinematics:
 Frequency f       : 0.18854 Hz
 Angular freq ω    : 1.18464 rad/s
 Period T          : 5.30387 s
--------------------------------------------------------
Structural Properties:
 Mass (m)          : 5.00000   [Expected: ~5.0 (Standard low mass ratio)]
 Damping Ratio (ζ) : 0.01000   [Expected: ~0.01 (Standard low damping)]
 Stiffness (k)     : 7.01689   [Derived from m * ω^2]
 Damping (c)       : 0.11846   [Derived from 2 * ζ * m * ω]
--------------------------------------------------------
Wake Oscillator Parameters (Facchinetti VIV Model):
 Epsilon (ε)       : 0.00120   [Expected: ~0.05 to 0.5. Controls nonlinear saturation]
 Coupling (A)      : -7.64840  [Expected: ~5 to 20. Scales structural acceleration effect]
--------------------------------------------------------
Galloping Lift Coefficients (Quasi-Steady Polynomial):
 a1 (Linear)       : 4.56276   [Expected: > 0. Positive value triggers soft galloping]
 a3 (C